In [7]:
import pandas as pd

df = pd.read_csv("../data/processed/predicted_threshold_equality_movie.csv")

df = df[df["model"] != "SemHDC_FastText_300"]

# Compute F1 per row (guard against zero denominator)
df["f1"] = df.apply(
    lambda r: (2 * r["precision"] * r["recall"]) / (r["precision"] + r["recall"])
    if (r["precision"] + r["recall"]) > 0 else 0.0,
    axis=1
)

avg_f1_pivot = (
    df.groupby(["m", "n", "model"])["f1"]
    .mean()
    .reset_index()
    .pivot(index=["m", "n"], columns="model", values="f1")
    .rename_axis(columns=None)
    .sort_index()
)

print(avg_f1_pivot.to_string())

     HRR_1024  HRR_2048   HRR_300   HRR_512
m n                                        
1 1  1.000000  1.000000  1.000000  1.000000
2 1  1.000000  1.000000  1.000000  1.000000
  2  1.000000  1.000000  1.000000  1.000000
3 1  1.000000  1.000000  1.000000  1.000000
  2  1.000000  1.000000  1.000000  1.000000
  3  1.000000  1.000000  1.000000  1.000000
4 1  1.000000  0.996102  0.999665  0.998109
  2  1.000000  0.967327  1.000000  1.000000
  3  1.000000  1.000000  1.000000  1.000000
  4  1.000000  1.000000  1.000000  1.000000
5 1  0.984793  0.997626  0.999161  0.996053
  2  1.000000  0.975362  0.993200  0.998461
  3  1.000000  1.000000  1.000000  1.000000
  4  1.000000  1.000000  1.000000  1.000000
  5  1.000000  1.000000  1.000000  1.000000
6 1  0.996951  0.988311  0.948315  0.997446
  2  1.000000  0.991534  0.924912  0.988591
  3  0.998060  1.000000  0.805150  0.999832
  4  1.000000  1.000000  1.000000  1.000000
  5  1.000000  1.000000  1.000000  1.000000
  6  1.000000  1.000000  1.00000

In [ ]:
df = pd.read_csv("../data/processed/threshold_equality_movie.csv")

TARGET_MODEL = "EmbDI_300"  # <-- change as needed

model_df = df[df["model"] == TARGET_MODEL].copy()

# Compute F1 per row (guard against zero denominator)
model_df["f1"] = model_df.apply(
    lambda r: (2 * r["precision"] * r["recall"]) / (r["precision"] + r["recall"])
    if (r["precision"] + r["recall"]) > 0 else 0.0,
    axis=1
)

# Step 1: average F1 per (m, n, t) triple for this model
avg_per_t = (
    model_df.groupby(["m", "n", "t"])["f1"]
    .mean()
    .reset_index()
    .rename(columns={"f1": "avg_f1_at_t"})
)

# Step 2: take the maximum across all t values for each (m, n)
embdi300 = (
    avg_per_t.groupby(["m", "n"])
    .apply(lambda g: g.loc[g["avg_f1_at_t"].idxmax()])
    .reset_index(drop=True)
    [["m", "n", "t", "avg_f1_at_t"]]
    .rename(columns={"t": "best_t", "avg_f1_at_t": "Max F1 EmbDI300"})
    .sort_values(["m", "n"])
)
embdi300.reset_index().pivot(index=["m", "n"], columns="best_t", values="Max F1 EmbDI300").rename_axis(columns=None)
embdi300 = embdi300.drop(columns=["best_t"])

print(f"Model: {TARGET_MODEL}\n")
print(embdi300.to_string(index=False))



Model: EmbDI_300

   m    n  Max F1 EmbDI 300
 1.0  1.0          1.000000
 2.0  1.0          1.000000
 2.0  2.0          0.833333
 3.0  1.0          0.984393
 3.0  2.0          0.980952
 3.0  3.0          1.000000
 4.0  1.0          0.803922
 4.0  2.0          0.737671
 4.0  3.0          1.000000
 4.0  4.0          1.000000
 5.0  1.0          0.756093
 5.0  2.0          0.722150
 5.0  3.0          0.700000
 5.0  4.0          0.900000
 5.0  5.0          0.800000
 6.0  1.0          0.726400
 6.0  2.0          0.854424
 6.0  3.0          0.852381
 6.0  4.0          1.000000
 6.0  5.0          0.771106
 6.0  6.0          0.909524
 7.0  1.0          0.680717
 7.0  2.0          0.736050
 7.0  3.0          0.740000
 7.0  4.0          0.866667
 7.0  5.0          0.866667
 7.0  6.0          0.816291
 7.0  7.0          0.849624
 8.0  1.0          0.678199
 8.0  2.0          0.531481
 8.0  3.0          0.526707
 8.0  4.0          0.760240
 8.0  5.0          0.672981
 8.0  6.0          0.800000
 8

/var/folders/xc/clb9hqz905359_dr6745071h0000gp/T/ipykernel_89486/192085245.py:25: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.loc[g["avg_f1_at_t"].idxmax()])


In [1]:
import pandas as pd

def compute_f1(df):
    return df.apply(
        lambda r: (2 * r["precision"] * r["recall"]) / (r["precision"] + r["recall"])
        if (r["precision"] + r["recall"]) > 0 else 0.0,
        axis=1
    )

# --- Table 1: average F1 per (m, n, model), models as columns ---
df1 = pd.read_csv("../data/processed/predicted_threshold_equality_movie.csv")
df1["f1"] = compute_f1(df1)
df1 = df1[df1["model"] != "SemHDC_FastText_300"]  # Exclude this model as per original code

table1 = (
    df1.groupby(["m", "n", "model"])["f1"]
    .mean()
    .reset_index()
    .pivot(index=["m", "n"], columns="model", values="f1")
    .rename_axis(columns=None)
    .sort_index()
)

# --- Table 2: max average F1 over t, per (m, n) for a target model ---
TARGET_MODEL1 = "EmbDI_300"  # <-- change as needed

df2 = pd.read_csv("../data/processed/threshold_equality_movie.csv")
df2["f1"] = compute_f1(df2)

table2 = (
    df2[df2["model"] == TARGET_MODEL1]
    .groupby(["m", "n", "t"])["f1"]
    .mean()
    .reset_index()
    .pipe(lambda g: g.loc[g.groupby(["m", "n"])["f1"].idxmax()])
    .set_index(["m", "n"])[["f1"]]
    .rename(columns={"f1": TARGET_MODEL1})
)


# --- Table 2: max average F1 over t, per (m, n) for a target model ---
TARGET_MODEL2 = "EmbDI_512"  # <-- change as needed

df3 = pd.read_csv("../data/processed/threshold_equality_movie.csv")
df3["f1"] = compute_f1(df3)

table3 = (
    df3[df3["model"] == TARGET_MODEL2]
    .groupby(["m", "n", "t"])["f1"]
    .mean()
    .reset_index()
    .pipe(lambda g: g.loc[g.groupby(["m", "n"])["f1"].idxmax()])
    .set_index(["m", "n"])[["f1"]]
    .rename(columns={"f1": TARGET_MODEL2})
)

# --- Join ---
combined = table1.join(table2, how="left")
combined = combined.join(table3, how="left")
combined = combined[["HRR_300", "HRR_512", "HRR_1024", TARGET_MODEL1, TARGET_MODEL2]]
# combined = combined.round(2)

print(combined.to_string())

        HRR_300   HRR_512  HRR_1024  EmbDI_300  EmbDI_512
m  n                                                     
1  1   1.000000  1.000000  1.000000   1.000000   1.000000
2  1   1.000000  0.998611  1.000000   1.000000   1.000000
   2   1.000000  1.000000  1.000000   0.833333   0.900000
3  1   1.000000  0.998291  0.986408   0.984393   0.988268
   2   1.000000  1.000000  1.000000   0.980952   0.946185
   3   1.000000  1.000000  1.000000   1.000000   1.000000
4  1   0.994787  0.945608  0.989912   0.803922   0.794180
   2   0.979133  1.000000  1.000000   0.737671   0.708472
   3   1.000000  1.000000  1.000000   1.000000   1.000000
   4   1.000000  1.000000  1.000000   1.000000   1.000000
5  1   0.963636  0.970300  0.993662   0.756093   0.725360
   2   1.000000  0.933333  0.996296   0.722150   0.733333
   3   1.000000  1.000000  1.000000   0.700000   0.933333
   4   1.000000  1.000000  1.000000   0.900000   0.900000
   5   1.000000  1.000000  1.000000   0.800000   0.866667
6  1   0.97211

In [2]:
combined_avg_m = (
    combined
    .groupby("m")
    .mean()
    .round(2)
)
print(combined_avg_m.to_string())
print(combined_avg_m.to_latex(float_format="%.2f"))

    HRR_300  HRR_512  HRR_1024  EmbDI_300  EmbDI_512
m                                                   
1      1.00     1.00      1.00       1.00       1.00
2      1.00     1.00      1.00       0.92       0.95
3      1.00     1.00      1.00       0.99       0.98
4      0.99     0.99      1.00       0.89       0.88
5      0.99     0.98      1.00       0.78       0.83
6      0.97     0.99      0.99       0.85       0.83
7      0.94     0.99      1.00       0.79       0.79
8      0.90     0.99      0.99       0.71       0.75
9      0.90     0.97      0.99       0.66       0.65
10     0.85     0.97      0.99       0.71       0.69
11     0.85     0.97      0.99       0.77       0.80
12     0.84     0.93      0.99       0.72       0.74
13     0.85     0.92      0.98       0.79       0.79
14     0.85     0.92      0.98       0.79       0.79
15     0.82     0.92      0.99       0.76       0.82
\begin{tabular}{lrrrrr}
\toprule
 & HRR_300 & HRR_512 & HRR_1024 & EmbDI_300 & EmbDI_512 \\
m &  & 